In [1]:
from openai import OpenAI
from pydantic_ai import Agent
from pydantic_ai import messages as m

openai_client = OpenAI()

In [2]:
instructions = """You are an expert assistant specialized in Antimicrobial Resistance (AMR) give citations for your answers and use given tools"""

In [4]:
user_prompt = "What is AMR?"

In [5]:

messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": user_prompt},
]

agent_research = Agent(
    name="research_agent",
    model="gpt-4o-mini",
    instructions=instructions 
)

In [6]:
response = await agent_research.run(user_prompt)

In [7]:
response.all_messages()

[ModelRequest(parts=[UserPromptPart(content='What is AMR?', timestamp=datetime.datetime(2026, 3, 29, 16, 57, 18, 874135, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 3, 29, 16, 57, 18, 874297, tzinfo=datetime.timezone.utc), instructions='You are an expert assistant specialized in Antimicrobial Resistance (AMR) give citations for your answers and use given tools', run_id='5850c85a-c7c5-496f-bfcd-c4bd29a10ecc'),
 ModelResponse(parts=[TextPart(content='Antimicrobial Resistance (AMR) occurs when microorganisms such as bacteria, viruses, fungi, and parasites evolve and develop the ability to resist the effects of medicines that once killed them or inhibited their growth. This resistance leads to the ineffectiveness of standard treatments, making infections harder to treat and increasing the risk of disease spread, severe illness, and death.\n\nAMR can occur naturally over time, through genetic changes in microorganisms. However, the misuse and overuse of antimicrobials

## RAG Testing 

In [8]:
# Reusable helper to query amr_knowledge_base from notebook
import chromadb
from pathlib import Path


COLLECTION_NAME = "amr_knowledge_base"
PATH_TO_CHROMA_DB = "../data/chroma_db"

def get_amr_collection() -> tuple[Path, chromadb.Collection]:
    # Try common locations and only accept a directory that already has the collection.
    chroma_dir = PATH_TO_CHROMA_DB
    checked: list[str] = []
    
    checked.append(f"{chroma_dir} (missing)")

    client = chromadb.PersistentClient(path=chroma_dir)
    try:
        collection = client.get_collection(COLLECTION_NAME)
        return chroma_dir, collection
    except Exception:
        checked.append(f"{chroma_dir} (collection missing)")

    checked_paths = ", ".join(checked)
    raise RuntimeError(
        f"Collection '{COLLECTION_NAME}' not found. Checked: {checked_paths}. "
        "Run ingestion first: uv run python scripts/ingest.py"
    )


def query_amr(question: str, top_k: int = 3) -> None:
    """
    Query the AMR knowledge base ChromaDB collection using a natural language question.

    Args:
        question: The user question to retrieve relevant AMR document chunks.
        top_k: The number of top similar results to return.

    Returns:
        None. Prints matching document chunks and their metadata, and returns the raw results dict.

    Example:
        >>> query_amr("What are the main risks of AMR?", top_k=5)
    """
    chroma_dir, collection = get_amr_collection()
    print(f"Using ChromaDB at: {chroma_dir}")
    print(f"Collection size: {collection.count()}")

    results = collection.query(
        query_texts=[question],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    for i, (doc, meta, dist) in enumerate(
        zip(results["documents"][0], results["metadatas"][0], results["distances"][0]),
        start=1,
    ):
        print(f"[{i}] distance={dist:.3f} source={meta.get('source_id', 'unknown')}")
        print(doc[:250].replace("\n", " "))
        print("---")
    return results


results = query_amr("what are the main risks of AMR?", top_k=10)


Using ChromaDB at: ../data/chroma_db
Collection size: 22860
[1] distance=0.704 source=lancet-gram-pmc_0001_pmc8841637
s study is the first to quantify the burden of AMR using two different AMR counterfactual scenarios.  **Implications of all the available evidence**
---
[2] distance=0.704 source=lancet-gram-pmc_0032_pmc8841637
s study is the first to quantify the burden of AMR using two different AMR counterfactual scenarios.  **Implications of all the available evidence**
---
[3] distance=0.704 source=lancet-gram-pmc_0034_pmc8841637
s study is the first to quantify the burden of AMR using two different AMR counterfactual scenarios.  **Implications of all the available evidence**
---
[4] distance=0.717 source=uk-amr-plan_0001_confronting-antimicrobial-resistance-2024-to-2029
nd accessibility issues.  5. What are the basic drivers and effects of AMR , and how does it spread?  This priority focuses on strengthening understanding of the basic drivers of AMR , including how it spreads and 

In [9]:
results

{'ids': [['c805d00a1fb93e325cbaf91589e4582e916957d04b5ee82f0d75a4afc470e8f9',
   'e528c7a038505b1874b09d3e16549dd1438b5c68d1227f2f75a61b9422aefbb9',
   'dfbe6fc827ea1dd77d1cd6557d3fa086fc6cca1018d7c7f053f6a75e1a827d4d',
   '287e8d3cfee912bb3af6c18d4fa8ff0d69b8ec4490a9025ff898360bb534f830',
   'a593f19fb0e11aec8dd5565e4b1c417dd9b527d5204f5a5f7198b590a1513f8c',
   '803576e98bdda9d913c616d173a683b1451ef15799f3594e456be2e7236cd503',
   'f849bcac4862f6f50ca415097de973b81a265ed18cdcb0f96ce6df91af1f19a7',
   'c4a17740c91ac99214a05c5156c7d3d567ec29f3194e9c60c0dd0dc06caaff1a',
   '21bcc6cfcf7cd768c379642ebd52c45fb4a6c1395a457795011daf571cd62f89',
   'f5168a0c4cee0b0290879fcaff51ff90357d6cbb13f2cc9893b26b1073077b82']],
 'embeddings': None,
 'documents': [['s study is the first to quantify the burden of AMR using two different AMR counterfactual scenarios.\n\n**Implications of all the available evidence**',
   's study is the first to quantify the burden of AMR using two different AMR counterfact

In [10]:
instructions = """You are an expert assistant specialized in Antimicrobial Resistance (AMR).

Your task is to answer the user’s query using provided context chunks or perform a web search if the question if you can access local files. 
Each chunk includes:
- chunk_id
- source_id
- text

---------------------
USER QUERY:
{user_query}

---------------------
CONTEXT CHUNKS:
{retrieved_chunks}

---------------------

INSTRUCTIONS:is not related to AMR

1. Provide a clear and concise DESCRIPTION:
   - A short summary (2–4 sentences) that directly answers the query.

2. Provide a DETAILED ANSWER:
   - Expand thoroughly using the context.
   - Explain mechanisms, causes, impacts, or examples where relevant.
   - Maintain scientific accuracy and clarity.

3. CITATIONS (MANDATORY):
   - Every key statement must include inline citations.
   - Use this exact format:
     [chunk_id: <chunk_id>, source_id: <source_id>]
   - If multiple chunks support a statement, cite all of them.
   - If you can access local files, use the web search tool to find the information.Give URL of the page.

4. STRICT GROUNDING:
   - If the answer is not in the context, say:
     "The provided context does not contain sufficient information."

5. STRUCTURE YOUR OUTPUT EXACTLY AS:

DESCRIPTION:
<short summary>

DETAILED ANSWER:
<fully developed answer with inline citations>

CITATIONS USED:
- chunk_id: X, source_id: Y
- chunk_id: A, source_id: B

6. STYLE:
   - Formal, scientific tone.
   - Avoid speculation.
   - Be precise and unambiguous.
 """

In [11]:
user_prompt = "What are the main risks of AMR?"

In [12]:
agent_research_rag = Agent(name="research_agent_rag",
                     model="gpt-4o-mini",
                     instructions=instructions,
                     tools=[query_amr])

response_rag = await agent_research_rag.run(user_prompt)
response= await agent_research.run(user_prompt)


Using ChromaDB at: ../data/chroma_db
Collection size: 22860
[1] distance=0.406 source=fao-amr_0001_en
ability to tolerate the treatments by becoming resistant, making the drugs less effective over time.  Antimicrobial resistance (AMR) is a major global threat of increasing concern to human and animal health. It also has implications for food safety, 
---
[2] distance=0.406 source=fao-amr_0015_en
ability to tolerate the treatments by becoming resistant, making the drugs less effective over time.  Antimicrobial resistance (AMR) is a major global threat of increasing concern to human and animal health. It also has implications for food safety, 
---
[3] distance=0.406 source=fao-amr_0016_zh
ability to tolerate the treatments by becoming resistant, making the drugs less effective over time.  Antimicrobial resistance (AMR) is a major global threat of increasing concern to human and animal health. It also has implications for food safety, 
---


In [13]:
response_rag.all_messages()

[ModelRequest(parts=[UserPromptPart(content='What are the main risks of AMR?', timestamp=datetime.datetime(2026, 3, 29, 16, 57, 25, 902134, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 3, 29, 16, 57, 25, 902305, tzinfo=datetime.timezone.utc), instructions='You are an expert assistant specialized in Antimicrobial Resistance (AMR).\n\nYour task is to answer the user’s query using provided context chunks or perform a web search if the question if you can access local files. \nEach chunk includes:\n- chunk_id\n- source_id\n- text\n\n---------------------\nUSER QUERY:\n{user_query}\n\n---------------------\nCONTEXT CHUNKS:\n{retrieved_chunks}\n\n---------------------\n\nINSTRUCTIONS:is not related to AMR\n\n1. Provide a clear and concise DESCRIPTION:\n   - A short summary (2–4 sentences) that directly answers the query.\n\n2. Provide a DETAILED ANSWER:\n   - Expand thoroughly using the context.\n   - Explain mechanisms, causes, impacts, or examples where relevant.\n   

In [14]:
pprint(response_rag.output)

Pretty printing has been turned OFF


In [15]:
response.all_messages()

[ModelRequest(parts=[UserPromptPart(content='What are the main risks of AMR?', timestamp=datetime.datetime(2026, 3, 29, 16, 57, 34, 384347, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 3, 29, 16, 57, 34, 384485, tzinfo=datetime.timezone.utc), instructions='You are an expert assistant specialized in Antimicrobial Resistance (AMR) give citations for your answers and use given tools', run_id='99e9622a-8451-40f9-b17a-8d500ed33b82'), ModelResponse(parts=[TextPart(content="Antimicrobial resistance (AMR) poses significant risks to public health, healthcare systems, and global economies. Here are the main risks associated with AMR:\n\n1. **Increased Mortality and Morbidity**: Infections caused by resistant pathogens can lead to higher mortality rates. Patients with resistant infections often have worse health outcomes compared to those with infections treated with effective antimicrobials (O'Neill, 2016).\n\n2. **Longer Hospital Stays**: Patients with AMR infections tend 

In [16]:
pprint(response.output)

Pretty printing has been turned ON


In [ ]:
# The exact request sent right before final answer
msgs = response_rag.all_messages()

final_request = [x for x in msgs if isinstance(x, m.ModelRequest)][-1]
print("Instructions:", final_request.instructions)
print("Parts in final request:", [type(p).__name__ for p in final_request.parts])
tool_parts = [p for p in final_request.parts if isinstance(p, m.ToolReturnPart)]
print("Tool returns in final request:", len(tool_parts))
if tool_parts:
  print("Tool:", tool_parts[0].tool_name)
  print(tool_parts[0].content)  

Instructions: You are an expert assistant specialized in Antimicrobial Resistance (AMR).

Your task is to answer the user’s query using provided context chunks or perform a web search if the question if you can access local files. 
Each chunk includes:
- chunk_id
- source_id
- text

---------------------
USER QUERY:
{user_query}

---------------------
CONTEXT CHUNKS:
{retrieved_chunks}

---------------------

INSTRUCTIONS:is not related to AMR

1. Provide a clear and concise DESCRIPTION:
   - A short summary (2–4 sentences) that directly answers the query.

2. Provide a DETAILED ANSWER:
   - Expand thoroughly using the context.
   - Explain mechanisms, causes, impacts, or examples where relevant.
   - Maintain scientific accuracy and clarity.

3. CITATIONS (MANDATORY):
   - Every key statement must include inline citations.
   - Use this exact format:
     [chunk_id: <chunk_id>, source_id: <source_id>]
   - If multiple chunks support a statement, cite all of them.
   - If you can acces

NameError: name 'pprint' is not defined

In [ ]:
# Implement human in the loop basically run until no tools are usedç
# Return the final answer and the citations and wait for user input to continue

async def run_agent_with_human_in_the_loop(agent, user_prompt):
    response = await agent.run(user_prompt)
    messages = response.all_messages()
    while True:
        print(response.output)
        user_input = input("Do you want to continue? (y/n)")
        if user_input == "n":
            break
        response = await agent.run(user_input, message_history=messages)
        messages.extend(response.all_messages())
    return response


# Think about the structured outputs of free form text? 

In [ ]:
response_new = await run_agent_with_human_in_the_loop(agent_research, "What are the main risks of AMR?")
response_new.all_messages()
print(response_new.output)

Antimicrobial resistance (AMR) poses several critical risks that can significantly impact public health, healthcare systems, and global economies. Here are the main risks associated with AMR:

1. **Increased Mortality and Morbidity**: Infections caused by resistant pathogens can lead to higher mortality rates. Patients with drug-resistant infections may suffer longer illnesses, leading to higher rates of hospitalization and increased severity of disease (World Health Organization, 2021).

2. **Limited Treatment Options**: As bacteria become resistant to commonly used antibiotics, treatment options become limited. This situation complicates the management of infections, leading to the need for more potent and often more toxic drugs, which may have higher side effects (Centers for Disease Control and Prevention, 2019).

3. **Prolonged Hospital Stays**: Patients with AMR infections often require longer hospital stays and more intensive care. This prolongs recovery times and increases heal

NameError: name 'pprint' is not defined

In [24]:
print(response_new.output)

I apologize if my previous response didn't meet your expectations. Could you please clarify your query regarding qgrams and antimicrobial resistance? Are you looking for a specific aspect or application of this concept? Your input will help me provide a more relevant response. Thank you!
